In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("../extraction/clientRaw.csv", sep=";")

df

In [ ]:
def eda_summary(df):
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "count": df.count(),
        "missing": df.isna().sum(),
        "missing_%": df.isna().mean() * 100,
        "unique": df.nunique(dropna=True),
    })

    summary["min"] = df.select_dtypes(include="number").min()
    summary["mean"] = df.select_dtypes(include="number").mean()
    summary["median"] = df.select_dtypes(include="number").median()
    summary["max"] = df.select_dtypes(include="number").max()

    return summary.round(2)

summary = eda_summary(df)
summary.to_csv("summary.csv", sep=";")

In [ ]:
summary

In [ ]:
dtypes = summary["dtype"].astype(str).value_counts()

plt.bar(
    dtypes.index,
    dtypes.values,
    edgecolor="black"
)

plt.title("Distribution of Data Types in Features")
plt.xlabel("Data Type")
plt.ylabel("Number of Columns")

plt.yticks(
    np.arange(0, dtypes.max() + 5, 10)
)

plt.tight_layout()
plt.show()

In [ ]:
plt.hist(summary["missing_%"], bins=10, edgecolor='black')
plt.title("Distribution of Missing Values (%) in features")
plt.xlabel("Percentage of Missing Values")
plt.ylabel("Number of Columns")
plt.xticks(np.arange(0, 101, 10))
plt.yticks(np.arange(0, 200, 10))
plt.show()

In [ ]:
df_dropped = df.drop(columns = summary[summary["missing_%"] >= 10].index)

In [ ]:


plt.hist(summary[(summary["missing_%"] < 10) & (summary["dtype"] == "object")]["unique"], bins=40, edgecolor='black')
plt.title("Distribution of unique values in 'object'-dtype columns")
plt.xlabel("Number of unique values")
plt.ylabel("Number of Features")

plt.show()

plt.hist(summary[(summary["missing_%"] < 10) & (summary["dtype"] == "object") & (summary["unique"] <= 1000)]["unique"], bins=10, edgecolor='black')
plt.title("Distribution of unique values in 'object'-dtype columns")
plt.xlabel("Number of unique values")
plt.xticks(np.arange(0,17,3))
plt.xlim(-1,18)
plt.ylabel("Number of Features")
plt.show()

In [ ]:
summary[(summary["missing_%"] < 10) & (summary["dtype"] == "object") & (summary["unique"] <= 100)]

In [ ]:
df_dropped["genderId"] = df_dropped["genderId"].where(
    df_dropped["genderId"].isin(["M", "F"]),
    "fam"
)

In [ ]:
df_dropped["admin-packet"].unique()

In [ ]:
from config import pflege_map

average = sum(pflege_map.values()) / len(pflege_map)

In [ ]:
df_dropped["admin-packet"] = (
    df_dropped["admin-packet"]
    .map(pflege_map)
    .fillna(average)
)

In [ ]:
df_dropped["admin-packet"].unique()

In [ ]:
columns_to_encode = [
    "genderId"
]

df_encoded = pd.get_dummies(
    df_dropped,
    columns=columns_to_encode,
    dtype=int,
	drop_first=True
)

In [ ]:
df_encoded

In [ ]:

df_encoded["dateOfBirth"] = pd.to_datetime(
    df_encoded["dateOfBirth"].astype("Int64").astype("string"),
    format="%Y%m%d",
    errors="coerce"
)



In [ ]:
df_encoded

In [ ]:
feature_summary = pd.DataFrame({
    "nunique": df_encoded.nunique(),
    "most_common_pct": df_encoded.apply(
        lambda col: col.value_counts(normalize=True, dropna=False).iloc[0] * 100
    )
})

print(feature_summary.sort_values("most_common_pct", ascending=False))

plt.hist(feature_summary["most_common_pct"], bins= 40, edgecolor="black")
plt.title("Distribution of Most Common Value Percentage Across Features")
plt.xlabel("Most Common Value (%)")
plt.ylabel("Number of Features")
plt.show()

In [ ]:
df_y = df_encoded

In [ ]:

df_y["birthYear"] = df_y["dateOfBirth"].dt.year
df_y["birthMonth"] = df_y["dateOfBirth"].dt.month



df_y

In [ ]:
df_y.drop(columns="dateOfBirth", inplace=True)

In [ ]:
df_final = df_y.drop(columns=feature_summary[feature_summary["most_common_pct"]>99].index)
df_final

In [ ]:
corr = df_final.corr(numeric_only=True)

# Hide upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Dynamic figure size
n_features = len(corr.columns)
figsize = max(12, n_features * 0.45)

fig, ax = plt.subplots(figsize=(figsize, figsize))

# Mask upper triangle
corr_masked = np.ma.masked_where(mask, corr.values)

im = ax.imshow(
    corr_masked,
    vmin=-1,
    vmax=1,
    aspect="equal"
)

fig.colorbar(im, ax=ax, label="Correlation")

ax.set_xticks(range(n_features))
ax.set_yticks(range(n_features))

ax.set_xticklabels(
    corr.columns,
    rotation=90,
    fontsize=8
)

ax.set_yticklabels(
    corr.columns,
    fontsize=8
)

ax.set_title(
    "Correlation Matrix of Model Features",
    fontsize=16,
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
corr = df_final.corr(numeric_only=True)

threshold = 0.8

strong_corr = corr.where(
    (corr.abs() >= threshold) & (corr.abs() < 1)
)

# Remove features that have no strong correlations
keep = strong_corr.notna().any()

strong_corr = strong_corr.loc[keep, keep]

In [ ]:
corr_matrix = strong_corr

plt.figure(figsize=(16, 12))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Correlation")

plt.xticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
    rotation=90,
	fontsize=16
)

plt.yticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
	fontsize=16
)

plt.title("Correlation Matrix of Model Features with absolut correlation higher than 0.8", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
def remove_correlated_features(df, threshold=0.8, exclude=["id"]):

    result = df.copy()
    dropped = []

    while True:

        features = result.drop(
            columns=exclude,
            errors="ignore"
        )

        corr = features.corr().abs()

        # Remove diagonal
        np.fill_diagonal(corr.values, 0)

        # Find strongest remaining correlation
        max_corr = corr.max().max()

        if max_corr <= threshold:
            break

        # Find the pair
        var1, var2 = corr.stack().idxmax()

        # Mean absolute correlation with all other variables
        score1 = corr.loc[var1].mean()
        score2 = corr.loc[var2].mean()

        # Drop the more redundant variable
        if score1 > score2:
            drop = var1
        else:
            drop = var2

        dropped.append({
            "dropped": drop,
            "var1": var1,
            "var2": var2,
            "pair_corr": max_corr,
            "score_var1": score1,
            "score_var2": score2
        })

        result = result.drop(columns=drop)

    return result, pd.DataFrame(dropped)

In [ ]:
df_final_cleaned, dropped =  remove_correlated_features(df_final)

dropped

In [ ]:
df_final_cleaned["genderId_M"] = df_final_cleaned["genderId_M"].astype("bool")
df_final_cleaned["genderId_fam"] = df_final_cleaned["genderId_M"].astype("bool")

In [ ]:
df_final_cleaned.to_csv("clients.csv", sep=";", index=False)